In [0]:
-- ============================================================
-- E-COMMERCE SALES ANALYTICS
-- DATABRICKS SQL PIPELINE
-- ============================================================
-- ============================================================
-- 1. SOURCE / CLEAN TABLE VALIDATION
-- ============================================================

USE CATALOG ecommerce_sales;
USE SCHEMA ecommerce;
-- ------------------------------------------------------------
-- 1.1 Check source Delta table
-- ------------------------------------------------------------

SELECT *
FROM ecommerce_sales.ecommerce.clean_sales
LIMIT 10;
-- ------------------------------------------------------------
-- 1.2 Check table structure
-- ------------------------------------------------------------

DESCRIBE ecommerce_sales.ecommerce.clean_sales;
-- ------------------------------------------------------------
-- 1.3 Basic data validation
-- ------------------------------------------------------------

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS missing_order_id,
    SUM(CASE WHEN product IS NULL THEN 1 ELSE 0 END) AS missing_product,
    SUM(CASE WHEN category IS NULL THEN 1 ELSE 0 END) AS missing_category,
    SUM(CASE WHEN quantity IS NULL THEN 1 ELSE 0 END) AS missing_quantity,
    SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS missing_price,
    SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS missing_city,
    SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS missing_date,
    SUM(CASE WHEN total_price IS NULL THEN 1 ELSE 0 END) AS missing_total_price
FROM ecommerce_sales.ecommerce.clean_sales;
-- ------------------------------------------------------------
-- 1.4 Check rescued records
-- ------------------------------------------------------------

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0
        END ) AS rescued_rows
FROM ecommerce_sales.ecommerce.clean_sales;

-- ------------------------------------------------------------
-- 1.5 Validate revenue calculation
-- ------------------------------------------------------------

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ABS(total_price - (quantity * price)) > 0.01 THEN 1 ELSE 0
        END ) AS incorrect_revenue_rows
FROM ecommerce_sales.ecommerce.clean_sales;
-- ------------------------------------------------------------
-- 1.6 Check invalid business values
-- ------------------------------------------------------------

SELECT
    SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END)AS invalid_quantity_rows,
    SUM(CASE WHEN price <= 0 THEN 1 ELSE 0 END)AS invalid_price_rows,
    SUM(CASE WHEN total_price <= 0 THEN 1 ELSE 0 END)AS invalid_revenue_rows
FROM ecommerce_sales.ecommerce.clean_sales;
-- ------------------------------------------------------------
-- 1.7 Check date range
-- ------------------------------------------------------------

SELECT
    MIN(date) AS first_order_date,
    MAX(date) AS last_order_date,
    COUNT(DISTINCT date) AS selling_days
FROM ecommerce_sales.ecommerce.clean_sales;
-- ------------------------------------------------------------
-- 1.8 Monthly distribution
-- ------------------------------------------------------------
SELECT
    year,month_number,month,COUNT(*) AS orders,
    SUM(quantity) AS units,
    ROUND(SUM(total_price), 2) AS revenue
FROM ecommerce_sales.ecommerce.clean_sales
GROUP BY year,month_number,month
ORDER BY year,month_number;
-- ============================================================
-- 2. CURATED TRANSFORMATION
-- ============================================================
-- Purpose:
-- Create a clean, analytics-ready Delta table with
-- explicitly controlled data types and standardized fields.

CREATE OR REPLACE TABLE ecommerce_sales.ecommerce.curated_sales
USING DELTA
AS
SELECT
    CAST(order_id AS STRING) AS order_id,
    TRIM(product) AS product,
    TRIM(category) AS category,
    CAST(quantity AS INT) AS quantity,
    CAST(price AS DECIMAL(12,2)) AS price,
    TRIM(city) AS city,
    CAST(date AS DATE) AS order_date,
    CAST(total_price AS DECIMAL(14,2)) AS total_price,
    CAST(year AS INT) AS year,
    CAST(month_number AS INT) AS month_number,
    CAST(DATE_TRUNC('MONTH', CAST(date AS DATE))AS DATE) AS month
FROM ecommerce_sales.ecommerce.clean_sales
WHERE order_id IS NOT NULL AND product IS NOT NULL
  AND category IS NOT NULL AND quantity > 0
  AND price > 0 AND total_price > 0
  AND date IS NOT NULL;
-- ------------------------------------------------------------
-- 2.1 Validate curated table
-- ------------------------------------------------------------

SELECT *
FROM ecommerce_sales.ecommerce.curated_sales
LIMIT 10;

DESCRIBE ecommerce_sales.ecommerce.curated_sales;
-- ------------------------------------------------------------
-- 2.2 Curated table row count
-- ------------------------------------------------------------

SELECT
    COUNT(*) AS curated_rows,
    COUNT(DISTINCT order_id) AS curated_orders,
    SUM(quantity) AS curated_units,
    ROUND(SUM(total_price), 2) AS curated_revenue
FROM ecommerce_sales.ecommerce.curated_sales;
-- ============================================================
-- 3. GOLD - MONTHLY SALES
-- ============================================================
-- Purpose:
-- Monthly business performance for Power BI and reporting.


CREATE OR REPLACE TABLE ecommerce_sales.ecommerce.gold_monthly_sales
USING DELTA
AS
SELECT
    year,month_number,month,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(quantity) AS total_units,
    ROUND(SUM(total_price), 2) AS total_revenue,
    ROUND(SUM(total_price)/ NULLIF(COUNT(DISTINCT order_id), 0),2) AS average_order_value,
    ROUND(SUM(total_price)/ NULLIF(SUM(quantity), 0),2) AS average_selling_price
FROM ecommerce_sales.ecommerce.curated_sales
GROUP BY year,month_number,month;
-- ------------------------------------------------------------
-- 3.1 Check Gold Monthly Sales
-- ------------------------------------------------------------

SELECT *
FROM ecommerce_sales.ecommerce.gold_monthly_sales
ORDER BY year,month_number;
-- ============================================================
-- 4. GOLD - PRODUCT PERFORMANCE
-- ============================================================
-- Purpose:
-- Product-level revenue, units, orders and rankings.
CREATE OR REPLACE TABLE ecommerce_sales.ecommerce.gold_product_performance
USING DELTA
AS
WITH product_metrics AS (

    SELECT product,category,COUNT(DISTINCT order_id) AS total_orders,
    SUM(quantity) AS total_units,
    ROUND(SUM(total_price), 2) AS total_revenue,
    ROUND(SUM(total_price)/ NULLIF(COUNT(DISTINCT order_id), 0),2) AS average_order_value,
    ROUND(SUM(total_price)/ NULLIF(SUM(quantity), 0),2) AS average_selling_price
    FROM ecommerce_sales.ecommerce.curated_sales
    GROUP BY product,category ),

ranked_products AS (
    SELECT
        *,
        RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
        RANK() OVER (ORDER BY total_units DESC) AS units_rank,
        ROUND(100.0 * total_revenue / 
            NULLIF(SUM(total_revenue) OVER (),0),2) AS revenue_contribution_pct
    FROM product_metrics
)

SELECT *
FROM ranked_products;
-- ------------------------------------------------------------
-- 4.1 Check Gold Product Performance
-- ------------------------------------------------------------
SELECT *
FROM ecommerce_sales.ecommerce.gold_product_performance
ORDER BY revenue_rank;
-- ============================================================
-- 5. GOLD - CATEGORY PERFORMANCE
-- ============================================================
-- Purpose:
-- Category-level business performance and contribution.


CREATE OR REPLACE TABLE ecommerce_sales.ecommerce.gold_category_performance
USING DELTA
AS
WITH category_metrics AS (
  SELECT
        category,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(quantity) AS total_units,
        ROUND(SUM(total_price), 2) AS total_revenue,
        ROUND(
            SUM(total_price)
            / NULLIF(COUNT(DISTINCT order_id), 0),
            2) AS average_order_value,
        ROUND(
            SUM(total_price)
            / NULLIF(SUM(quantity), 0),
            2) AS average_selling_price
    FROM ecommerce_sales.ecommerce.curated_sales
    GROUP BY category
),

category_ranked AS (
    SELECT
        *,
        RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
        ROUND(100.0 * total_revenue
            / NULLIF(
                SUM(total_revenue) OVER (),0),2) AS revenue_contribution_pct,
        ROUND(
            100.0 * total_units
            / NULLIF(
                SUM(total_units) OVER (),0),2) AS unit_contribution_pct
    FROM category_metrics
)

SELECT *
FROM category_ranked;
-- ------------------------------------------------------------
-- 5.1 Check Gold Category Performance
-- ------------------------------------------------------------

SELECT *
FROM ecommerce_sales.ecommerce.gold_category_performance
ORDER BY revenue_rank;
-- ============================================================
-- 6. GOLD - CITY PERFORMANCE
-- ============================================================
-- Purpose:
-- Geographic sales performance.

CREATE OR REPLACE TABLE ecommerce_sales.ecommerce.gold_city_performance
USING DELTA
AS
WITH city_metrics AS (
    SELECT
        city,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(quantity) AS total_units,
        ROUND(SUM(total_price), 2) AS total_revenue,
        ROUND(
            SUM(total_price)
            / NULLIF(COUNT(DISTINCT order_id), 0),2) AS average_order_value,
        ROUND(
            SUM(total_price)
            / NULLIF(SUM(quantity), 0),2) AS average_selling_price
    FROM ecommerce_sales.ecommerce.curated_sales
    GROUP BY city
),

city_ranked AS (

    SELECT
        *,
        RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
        RANK() OVER (ORDER BY total_orders DESC) AS order_rank,
        ROUND(
            100.0 * total_revenue
            / NULLIF(
                SUM(total_revenue) OVER (),0),2) AS revenue_contribution_pct
    FROM city_metrics
)

SELECT *
FROM city_ranked;
-- ------------------------------------------------------------
-- 6.1 Check Gold City Performance
-- ------------------------------------------------------------

SELECT *
FROM ecommerce_sales.ecommerce.gold_city_performance
ORDER BY revenue_rank;
-- ============================================================
-- 7. GOLD - TIME SERIES
-- ============================================================
-- Purpose:
-- Monthly trend analysis including:
-- revenue growth, cumulative revenue and revenue contribution.


CREATE OR REPLACE TABLE ecommerce_sales.ecommerce.gold_time_series
USING DELTA
AS
WITH monthly AS (

    SELECT
        year,month_number,month,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(quantity) AS total_units,
        ROUND(SUM(total_price), 2) AS total_revenue
    FROM ecommerce_sales.ecommerce.curated_sales
    GROUP BY  year,month_number,month
),

growth AS (

    SELECT
        *,
        LAG(total_revenue) OVER (ORDER BY year, month_number) AS previous_month_revenue
    FROM monthly
)

SELECT
    year,month_number,month,total_orders,total_units,total_revenue,previous_month_revenue,
    ROUND(
        CASE WHEN previous_month_revenue IS NULL
              OR previous_month_revenue = 0
            THEN NULL
            ELSE
                (
    (total_revenue - previous_month_revenue)/ previous_month_revenue) * 100
        END,2) AS mom_revenue_growth_pct,

    ROUND(
        SUM(total_revenue) OVER (
            ORDER BY year, month_number
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),2) AS cumulative_revenue,
    ROUND(
        100.0 * total_revenue
        / NULLIF(
            SUM(total_revenue) OVER (),0),2) AS revenue_contribution_pct
FROM growth;

-- ------------------------------------------------------------
-- 7.1 Check Gold Time Series
-- ------------------------------------------------------------
SELECT *
FROM ecommerce_sales.ecommerce.gold_time_series
ORDER BY year,month_number;
-- ============================================================
-- 8. FINAL VALIDATION
-- ============================================================
-- ------------------------------------------------------------
-- 8.1 Check all Gold tables
-- ------------------------------------------------------------

SELECT 'curated_sales' AS table_name,COUNT(*) AS row_count
FROM ecommerce_sales.ecommerce.curated_sales

UNION ALL

SELECT 'gold_monthly_sales',COUNT(*)
FROM ecommerce_sales.ecommerce.gold_monthly_sales

UNION ALL

SELECT 'gold_product_performance',COUNT(*)
FROM ecommerce_sales.ecommerce.gold_product_performance

UNION ALL

SELECT 'gold_category_performance',COUNT(*)
FROM ecommerce_sales.ecommerce.gold_category_performance

UNION ALL

SELECT 'gold_city_performance',COUNT(*)
FROM ecommerce_sales.ecommerce.gold_city_performance

UNION ALL

SELECT 'gold_time_series',COUNT(*)
FROM ecommerce_sales.ecommerce.gold_time_series;
-- ------------------------------------------------------------
-- 8.2 Final business KPI validation
-- ------------------------------------------------------------
SELECT
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(quantity) AS total_units,
    ROUND(SUM(total_price),2) AS total_revenue,
    ROUND(
        SUM(total_price)
        / NULLIF(COUNT(DISTINCT order_id), 0),2) AS average_order_value,
    ROUND(
        SUM(total_price)
        / NULLIF(SUM(quantity), 0),2) AS average_selling_price
FROM ecommerce_sales.ecommerce.curated_sales;
-- ------------------------------------------------------------
-- 8.3 Final date validation
-- ------------------------------------------------------------
SELECT
    MIN(order_date) AS first_order_date,
    MAX(order_date) AS last_order_date,
    COUNT(DISTINCT month) AS total_months
FROM ecommerce_sales.ecommerce.curated_sales;
-- ------------------------------------------------------------
-- 8.4 Final revenue consistency check
-- ------------------------------------------------------------
SELECT
    ROUND(SUM(quantity * price),2) AS calculated_revenue,
    ROUND(SUM(total_price),2) AS stored_revenue,
    ROUND(SUM(quantity * price)- SUM(total_price),2) AS revenue_difference
FROM ecommerce_sales.ecommerce.curated_sales;
-- ============================================================
-- END OF DATABRICKS SQL PIPELINE
-- ============================================================

SELECT * 
FROM ecommerce_sales.ecommerce.gold_monthly_sales
ORDER BY month;


SELECT
    product,category,total_orders,total_units,total_revenue,
    average_order_value,revenue_rank,units_rank,revenue_contribution_pct
FROM ecommerce_sales.ecommerce.gold_product_performance
ORDER BY revenue_rank;

SELECT *
FROM ecommerce_sales.ecommerce.gold_category_performance
ORDER BY revenue_rank;


-- ============================================================
-- GOLD - DAILY SALES
-- ============================================================
CREATE OR REPLACE TABLE ecommerce_sales.ecommerce.gold_daily_sales
USING DELTA
AS
SELECT
    order_date,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(quantity) AS total_units,
    ROUND(SUM(total_price), 2) AS total_revenue,
    ROUND(
        SUM(total_price) /NULLIF(COUNT(DISTINCT order_id), 0),2
    ) AS average_order_value,
    ROUND(
        SUM(total_price) /NULLIF(SUM(quantity), 0),2
    ) AS average_selling_price
FROM ecommerce_sales.ecommerce.curated_sales
GROUP BY order_date
ORDER BY order_date;

SELECT *
FROM ecommerce_sales.ecommerce.gold_daily_sales
ORDER BY order_date;

-- ============================================================
-- GOLD Sales Features
-- ============================================================

-- ============================================================
-- GOLD - SALES FEATURES FOR ML
-- ============================================================

CREATE OR REPLACE TABLE ecommerce_sales.ecommerce.gold_sales_features
USING DELTA
AS
SELECT
    order_date,
    DAY(order_date) AS day,
    DAYOFWEEK(order_date) AS day_of_week,
    WEEKOFYEAR(order_date) AS week,
    MONTH(order_date) AS month,
    QUARTER(order_date) AS quarter,

    CASE WHEN DAYOFWEEK(order_date) IN (1, 7)THEN 1 ELSE 0 END AS is_weekend,
    total_revenue,
    LAG(total_revenue, 1)OVER (ORDER BY order_date) AS revenue_lag_1,
    LAG(total_revenue, 7)OVER (ORDER BY order_date) AS revenue_lag_7,
    LAG(total_revenue, 14)OVER (ORDER BY order_date) AS revenue_lag_14,
    LAG(total_revenue, 30)OVER (ORDER BY order_date) AS revenue_lag_30,

    AVG(total_revenue) OVER (
            ORDER BY order_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS revenue_rolling_7,

    AVG(total_revenue) OVER (
            ORDER BY order_date
            ROWS BETWEEN 14 PRECEDING AND 1 PRECEDING) AS revenue_rolling_14,

    AVG(total_revenue) OVER (
            ORDER BY order_date
            ROWS BETWEEN 30 PRECEDING AND 1 PRECEDING) AS revenue_rolling_30
FROM ecommerce_sales.ecommerce.gold_daily_sales
ORDER BY order_date;

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN revenue_lag_1 IS NULL THEN 1 ELSE 0 END) AS missing_lag_1,
    SUM(CASE WHEN revenue_lag_7 IS NULL THEN 1 ELSE 0 END) AS missing_lag_7,
    SUM(CASE WHEN revenue_lag_14 IS NULL THEN 1 ELSE 0 END) AS missing_lag_14,
    SUM(CASE WHEN revenue_lag_30 IS NULL THEN 1 ELSE 0 END) AS missing_lag_30,
    SUM(CASE WHEN revenue_rolling_7 IS NULL THEN 1 ELSE 0 END) AS missing_rolling_7,
    SUM(CASE WHEN revenue_rolling_14 IS NULL THEN 1 ELSE 0 END) AS missing_rolling_14,
    SUM(CASE WHEN revenue_rolling_30 IS NULL THEN 1 ELSE 0 END) AS missing_rolling_30
FROM ecommerce_sales.ecommerce.gold_sales_features;

SELECT
    COUNT(*) AS selling_days,
    DATEDIFF(MAX(order_date), MIN(order_date)) + 1 AS calendar_days,
    (DATEDIFF(MAX(order_date), MIN(order_date)) + 1) - COUNT(*) AS missing_calendar_days
FROM ecommerce_sales.ecommerce.gold_daily_sales;

